In [11]:
# GPU-Accelerated Heart Disease – RAPIDS Experiments

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load same dataset as baseline

df = pd.read_csv("/kaggle/input/heart-disease-data/heart_disease_uci.csv")
print("Shape:", df.shape)
print(df.head(3))





Shape: (920, 16)
   id  age   sex    dataset              cp  trestbps   chol    fbs  \
0   1   63  Male  Cleveland  typical angina     145.0  233.0   True   
1   2   67  Male  Cleveland    asymptomatic     160.0  286.0  False   
2   3   67  Male  Cleveland    asymptomatic     120.0  229.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  


In [12]:
# Define feature sets (same as baseline notebook)
numeric_features = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

X = df.drop(columns=["num"])
y = (df["num"] > 0).astype(int)

print("Features:", X.shape, "Target distribution:", y.value_counts().to_dict())

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Features: (920, 15) Target distribution: {1: 509, 0: 411}
Train: (736, 15) Test: (184, 15)


In [13]:
# CPU RandomForest baseline – accuracy + timing

rf_cpu = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

t0 = time.perf_counter()
rf_cpu.fit(X_train, y_train)
t1 = time.perf_counter()

y_pred_cpu = rf_cpu.predict(X_test)
cpu_acc = accuracy_score(y_test, y_pred_cpu)

print("CPU RandomForest")
print(f"  Accuracy: {cpu_acc:.3f}")
print(f"  Train time: {t1 - t0:.4f} seconds")


CPU RandomForest
  Accuracy: 0.859
  Train time: 0.7211 seconds


In [14]:
# CPU RandomForest baseline – accuracy + timing

rf_cpu = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

t0 = time.perf_counter()
rf_cpu.fit(X_train, y_train)
t1 = time.perf_counter()

y_pred_cpu = rf_cpu.predict(X_test)
cpu_acc = accuracy_score(y_test, y_pred_cpu)

print("CPU RandomForest")
print(f"  Accuracy: {cpu_acc:.3f}")
print(f"  Train time: {t1 - t0:.4f} seconds")


CPU RandomForest
  Accuracy: 0.859
  Train time: 0.6573 seconds


In [15]:
# GPU RandomForest with cuML (numeric features only, with imputation)

try:
    from cuml.ensemble import RandomForestClassifier as cuRF  # type: ignore
    import cudf  # type: ignore
    import cupy as cp  # type: ignore
except ImportError:
    print("cuML / cuDF not available. Skip this cell.")
else:
    # Convert pandas to cuDF
    X_train_cudf = cudf.from_pandas(X_train[numeric_features])
    X_test_cudf = cudf.from_pandas(X_test[numeric_features])
    y_train_cudf = cudf.from_pandas(y_train.to_frame(name="target"))["target"]
    y_test_cudf = cudf.from_pandas(y_test.to_frame(name="target"))["target"]

    # GPU-side simple imputation: fill NaNs with column medians
    for col in numeric_features:
        median_val = X_train_cudf[col].median()
        X_train_cudf[col] = X_train_cudf[col].fillna(median_val)
        X_test_cudf[col] = X_test_cudf[col].fillna(median_val)

    rf_gpu = cuRF(
        n_estimators=300,
        max_depth=15,
        random_state=42,
    )

    t0 = time.perf_counter()
    rf_gpu.fit(X_train_cudf, y_train_cudf)
    t1 = time.perf_counter()

    y_pred_gpu = rf_gpu.predict(X_test_cudf).to_pandas()
    gpu_acc = accuracy_score(y_test, y_pred_gpu)

    print("GPU RandomForest (cuML, numeric features only)")
    print(f"  Accuracy: {gpu_acc:.3f}")
    print(f"  Train time: {t1 - t0:.4f} seconds")


GPU RandomForest (cuML, numeric features only)
  Accuracy: 0.739
  Train time: 0.5748 seconds


In [17]:
print("RESULTS SUMMARY")

try:
    print(f"CPU RandomForest (full pipeline)")
    print(f"  Accuracy: {cpu_acc:.3f}")
except NameError:
    print("  CPU RF metrics not available (run CPU cell first).")

try:
    print(f"\nGPU RandomForest (cuML, numeric features only)")
    print(f"  Accuracy: {gpu_acc:.3f}")
    print("  Note: lower accuracy is expected here because GPU model")
    print("        is using only numeric features and a simpler pipeline.")
except NameError:
    print("\nGPU RF metrics not available (run GPU cell first).")


RESULTS SUMMARY
CPU RandomForest (full pipeline)
  Accuracy: 0.859

GPU RandomForest (cuML, numeric features only)
  Accuracy: 0.739
  Note: lower accuracy is expected here because GPU model
        is using only numeric features and a simpler pipeline.
